In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Dict, Optional, Set
import re
import polars as pl
import json
from pathlib import Path
import pandas as pd
import matplotlib as plt
import seaborn as sns
import matplotlib as plt

In [69]:
BIG_DATA_DIR = Path("/Users/veronica/Library/CloudStorage/Dropbox/earnings_calls")
RAW_EARNINGS_PATH = Path("/Users/veronica/Downloads")
PROCESSED_EARNINGS_PATH = BIG_DATA_DIR / "processed_data"

BASE_DIR = Path.cwd().parent.parent
GITHUB_DATA_PATH = BASE_DIR / "data" 
OUTPUTS_PATH = BASE_DIR / "outputs"

# Code to clean the earnings calls

In [26]:
class ChunkType(Enum):
    OPENING = "opening_statement"
    QA = "q_and_a"

class ParseStatus(Enum):
    SUCCESS = "success"
    NO_QA_FOUND = "no_qa_found"
    EMPTY_TRANSCRIPT = "empty_transcript"
    PARSE_ERROR = "parse_error"

@dataclass
class TranscriptChunk:
    call_id: str
    chunk_id: int
    chunk_type: ChunkType
    
    content: List[str] = field(default_factory=list) 
    speakers: List[str] = field(default_factory=list)
    
    question_text: List[str] = field(default_factory=list)
    question_speakers: List[str] = field(default_factory=list)
    answer_text: List[str] = field(default_factory=list)
    answer_speakers: List[str] = field(default_factory=list)

    def to_dict(self):
        if self.chunk_type == ChunkType.OPENING:
            final_q = "N/A (Opening Statement)"
            final_q_speakers = "N/A"
            final_a = "\n".join(self.content)
            final_a_speakers = ", ".join(sorted(set(self.speakers)))
        else:
            final_q = "\n".join(self.question_text) if self.question_text else "[No question captured]"
            final_q_speakers = ", ".join(sorted(set(self.question_speakers))) if self.question_speakers else "Unknown"
            final_a = "\n".join(self.answer_text) if self.answer_text else "[No answer captured]"
            final_a_speakers = ", ".join(sorted(set(self.answer_speakers))) if self.answer_speakers else "Unknown"

        return {
            'id': self.call_id,
            'chunk_id': self.chunk_id,
            'chunk_type': self.chunk_type.value,
            'question': final_q,
            'question_speakers': final_q_speakers,
            'answer': final_a,
            'answer_speakers': final_a_speakers
        }

In [41]:
class EarningsCallTranscript:
    """
    Label the speaker and metadata for each earnings call.
    The class returns the identified characteristics for the Earnings Call
    """
    
    def __init__(self, call_id, raw_dict):
        self.call_id = call_id
        self.raw_dict = raw_dict
        
        # attributes
        self.cleaned_lines = []
        self.managers = set()
        self.start_QA = None
        self.parse_status = ParseStatus.SUCCESS
        self.parse_metadata = {}
        
    def _clean(self):
        """
        we need to clean the names of the people in the call
        fill in cleaned_lines
        """
        temp_items = []
        
        # pattern for the order of the earnings call
        seq_pattern = re.compile(r'\[(\d+)\]')
        # pattern for the speaker name
        name_clean_pattern = re.compile(r'\s*-\s*\[\d+\]|\[\d+\]|\,')

        for key, text in self.raw_dict.items():
            # get the seq to keep the order
            seq_match = seq_pattern.search(key)
            seq_num = int(seq_match.group(1)) if seq_match else 0
            
            # clean name - preserve company/title info - #TODO decide what i'm doing with the company
            clean_name = name_clean_pattern.sub('', key).strip()
            clean_text = text.strip()

            if not clean_text or "[_" in clean_text:
                continue

            # Flag unidentified speakers but keep them
            is_unidentified = "unidentified" in clean_name.lower()
            
            temp_items.append({
                'seq': seq_num,
                'speaker': clean_name,
                'text': clean_text,
                'is_operator': 'Operator' in clean_name or 'operator' in clean_name.lower(),
                'is_unidentified': is_unidentified
            })

        # populate cleaned lines
        self.cleaned_lines = sorted(temp_items, key=lambda x: x['seq'])

        if not self.cleaned_lines:
            self.parse_status = ParseStatus.EMPTY_TRANSCRIPT

        return self
    
    # identify the managers
    def _identify_managers_and_split(self):
        """
        Scans opening statements to find manager names. Individuals who speak after operator opens the call &
        Before the operator intervenes again to open the questions
        """
        call_opened = False
        qa_keywords = ['question', 'press', 'q&a', 'open the floor', 'take questions']

        for idx, line in enumerate(self.cleaned_lines):
            # operator should open within first 10 lines
            if not call_opened and line['seq'] > 10:
                # try to recover - maybe first speaker is operator, usual case
                if idx == 0:
                    call_opened = True
                    continue
                else:
                    self.parse_status = ParseStatus.PARSE_ERROR
                    self.parse_metadata['error'] = 'Operator opening not found'
                    return

            # operator opens the call
            if line['is_operator'] and not call_opened:
                call_opened = True
                continue

            # get q&a start
            if call_opened and line['is_operator']:
                text_lower = line['text'].lower()
                if any(kw in text_lower for kw in qa_keywords):
                    self.start_QA = line['seq']
                    break
            # managers speak during opening
            if call_opened and not line['is_operator'] and not line['is_unidentified']:
                self.managers.add(line['speaker'])
                
        self.parse_metadata['num_managers'] = len(self.managers)
        self.parse_metadata['qa_start_seq'] = self.start_QA

        if self.start_QA is None:
            self.parse_status = ParseStatus.NO_QA_FOUND

    # lets identify all speakers not just the managers 

    def _classify_speaker(self, speaker_name: str) -> str:
        """classify speaker as manager, analyst, or unknown"""
        if speaker_name in self.managers:
            return 'manager'
        
        speaker_lower = speaker_name.lower()
        
        # analyst indicators - they give us this in their names for older transcripts
        analyst_markers = ['analyst', 'research', 'capital', 'securities', 
                        'partners', 'bank', 'credit', 'investment', 'markets']
        
        if any(marker in speaker_lower for marker in analyst_markers):
            return 'analyst'
        
        if 'unidentified' in speaker_lower:
            return 'unidentified'
        
        return 'unknown'
    
    def _is_likely_question(self, text: str, speaker_type: str) -> bool:
        """
        Heuristic algorithm to detect if a statement is likely a question
        """
        text_lower = text.lower()
        
        # question indicators
        question_markers = ['?', 'could you', 'can you', 'would you', 'do you', 
                           'what', 'how', 'why', 'when', 'where', 'who',
                           'wondering if', 'want to ask', 'question about',
                           'just curious', "i'd like to know"]
        
        # greeting patterns
        greeting_patterns = ['good morning', 'good afternoon', 'thank you for taking',
                            'thanks for taking', 'appreciate', 'quick question', "thank you very much"]
        
        has_question_marker = any(marker in text_lower for marker in question_markers)
        has_greeting = any(pattern in text_lower for pattern in greeting_patterns)
        
        # if speaker is DEFINITELY not management, likely asking question
        is_external = speaker_type == 'analyst' or speaker_type == 'unidentified'
        
        # either has a question marker OR is both an external speaker AND has a greeting in it
        return has_question_marker or (is_external and has_greeting)


    def parse(self):
        """Parse transcript into structured chunks"""
        # clean chunk
        self._clean()
        if self.parse_status != ParseStatus.SUCCESS:
            return self._create_status_dataframe()
        
        # identify managers and Q&A boundary
        self._identify_managers_and_split()
        if self.parse_status != ParseStatus.SUCCESS:
            return self._create_status_dataframe()
        
        final_chunks = []
        
        # Opening statements chunk
        opening_chunk = TranscriptChunk(
            call_id=self.call_id,
            chunk_id=1,
            chunk_type=ChunkType.OPENING
        )
        
        # stop at the start of the Q&A
        for line in self.cleaned_lines:
            if line['seq'] >= self.start_QA:
                break
            # save if it's not the operator - not part of the peopel
            if not line['is_operator']:
                opening_chunk.content.append(f"{line['speaker']}: {line['text']}")
                opening_chunk.speakers.append(line['speaker'])
        
        if opening_chunk.content:
            final_chunks.append(opening_chunk.to_dict())
        
        # Q&A section if we identified and its not empty
        if self.start_QA:
            current_id = 2
            qa_chunk = TranscriptChunk(
                call_id=self.call_id,
                chunk_id=current_id,
                chunk_type=ChunkType.QA
            )
            
            in_answer_mode = False
            promoted_managers = set()  # were promoting people to managers bc not all of them speak in the beginning
            
            for line in self.cleaned_lines:
                if line['seq'] <= self.start_QA:
                    continue
                if line['is_operator']:
                    # operator announcing next questioner
                    continue
                
                speaker = line['speaker']
                text = line['text']
                full_text = f"{speaker}: {text}"
                speaker_type = self._classify_speaker(speaker)
                
                # Manager Speaks (Answer)
                if speaker in self.managers or speaker in promoted_managers:
                    qa_chunk.answer_text.append(full_text)
                    qa_chunk.answer_speakers.append(speaker)
                    in_answer_mode = True
                
                # Handling manager handoff -> peopel sometimes start intervening in the call
                # but they didn't do any talkign at the beginning
                elif speaker_type == 'unknown' and in_answer_mode and qa_chunk.answer_text:
                    # check if previous manager handed off in the last text
                    last_answer = qa_chunk.answer_text[-1] if qa_chunk.answer_text else ""
                    # handoff manager is named after
                    first_name = speaker.split()[0].lower() if speaker else ""
                    name_mentioned = first_name in last_answer.lower()
                    
                    handoff_keywords = ['take', 'respond', 'comment', 'add', 'peter', 'steve', 
                                       'elaborate', 'detail', 'handle', 'go ahead', 'thoughts']
                    # check that the last answer had the handoff keyword
                    has_handoff = any(kw in last_answer.lower() for kw in handoff_keywords)
                    
                    is_short_handoff = len(last_answer) < 250
                    
                    # so if the person was named AND the last answer was short AND there's handoff
                    if name_mentioned and has_handoff and is_short_handoff:
                        # promote spealer to manager
                        promoted_managers.add(speaker)
                        self.managers.add(speaker)
                        # add text to answer
                        qa_chunk.answer_text.append(full_text)
                        qa_chunk.answer_speakers.append(speaker)
                        continue
                
                # analyst/external speaking (question)
                if speaker_type in ['analyst', 'unidentified', 'unknown']:
                    # if we were in answer mode, save previous Q&A pair bc we enter question mode
                    if in_answer_mode:
                        if qa_chunk.question_text or qa_chunk.answer_text:
                            final_chunks.append(qa_chunk.to_dict())
                        
                        # start new Q&A pair
                        current_id += 1
                        qa_chunk = TranscriptChunk(
                            call_id=self.call_id,
                            chunk_id=current_id,
                            chunk_type=ChunkType.QA
                        )
                        in_answer_mode = False
                    
                    qa_chunk.question_text.append(full_text)
                    qa_chunk.question_speakers.append(speaker)
            
            # save the chunk we were studying to the list of chunks and move to the next line
            if qa_chunk.question_text or qa_chunk.answer_text:
                final_chunks.append(qa_chunk.to_dict())
        
        # store metadata for the call 
        self.parse_metadata['num_chunks'] = len(final_chunks)
        self.parse_metadata['num_qa_pairs'] = len([c for c in final_chunks if c['chunk_type'] == 'q_and_a'])
        
        return pl.DataFrame(final_chunks)
    
    def _create_status_dataframe(self):
        """Create a dataframe indicating parse failure"""
        return pl.DataFrame([{
            'id': self.call_id,
            'chunk_id': 0,
            'chunk_type': 'parse_failed',
            'question': f"Parse Status: {self.parse_status.value}",
            'question_speakers': str(self.parse_metadata),
            'answer': '',
            'answer_speakers': ''
        }])
    
    def get_parse_summary(self) -> Dict:
        """Get summary of parsing results"""
        return {
            'call_id': self.call_id,
            'status': self.parse_status.value,
            'managers_identified': list(self.managers),
            'num_managers': len(self.managers),
            **self.parse_metadata
            }

In [39]:
def parse_earnings_calls(call_data_dict: Dict[str, Dict]) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    parse multiple earnings calls and track success - make statistics of success by year
    
    Returns:
        chunks_df: DataFrame of all parsed chunks
        summary_df: DataFrame of parse summaries
    """
    all_chunks = []
    summaries = []
    # dict key is the id, dict value is the raw dictionary for the call
    for call_id, raw_dict in call_data_dict.items():
        parser = EarningsCallTranscript(call_id, raw_dict)
        chunks_df = parser.parse()
        
        all_chunks.append(chunks_df)
        summaries.append(parser.get_parse_summary())
    
    combined_chunks = pl.concat(all_chunks, how='vertical')
    summary_df = pl.DataFrame(summaries)
    
    return combined_chunks, summary_df

# Run over all the files from 2001 to 2020

In [71]:
# years = range(2001,2021)
years = [2001,2012]

cleaning_metrics = {
    'year': list(),
    'total_calls': list(),
    'success_rate': list()
    }

for year in years:
    with open(RAW_EARNINGS_PATH / f'transcripts_year{year}.json', 'r') as f:
        data = json.load(f)

    combined_chunks_call, summary_df_calls = parse_earnings_calls(data)

     # write the chunks out
    combined_chunks_call.write_parquet(PROCESSED_EARNINGS_PATH / f'cleaned_transcripts_{year}.parquet')

    # write out the success stats
    successful_calls = summary_df_calls.filter(pl.col('status') == 'success')
    success_rate = (len(successful_calls)/len(summary_df_calls))*100

    # rates of success to dict
    cleaning_metrics['year'].append(year)
    cleaning_metrics['total_calls'].append(len(summary_df_calls))
    cleaning_metrics['success_rate'].append(success_rate)

    # write out success metric
    print(f"=====Done with year {year}=====")
    print(f"Processed {len(combined_chunks_call)} earnings calls")
    print(f"Success rate {success_rate}% earnings calls")

# cleaning stats dump
with open(GITHUB_DATA_PATH / 'outputs/cleaning_metrics.json', 'w') as json_f:
    json.dump(cleaning_metrics, json_f)

=====Done with year 2001=====
Processed 8615 earnings calls
Success rate 84.1025641025641% earnings calls
=====Done with year 2012=====
Processed 169591 earnings calls
Success rate 73.22231303637106% earnings calls
